# TASK 3 — Descriptor Grouping & Aggregation

**Goal**: Make descriptor-level results interpretable at family level.

## Steps
1. Group descriptors into families by name:
   - **Size/Weight**: MolWt, HeavyAtomCount, etc.
   - **Surface Area / VSA**: LabuteASA, SMR_VSA*, PEOE_VSA*
   - **Topological**: Chi*, Kappa*
   - **Charge/Electronic**: MaxPartialCharge, MinPartialCharge, etc.
   - **Fragment Counts**: fr_*
   - **Rings**: NumAromaticRings, NumSaturatedRings, etc.
   - **Lipophilicity**: MolLogP, MolMR
   - **Other**: Remaining descriptors

2. For each family:
   - Report median and IQR of effect ratios
   - Count descriptors in family

3. Create summary table

## Outputs
- CSV: `results/indicators/descriptor_family_summary.csv`
- Console: Family-level summary table

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

✓ Imports complete


## 1. Load TASK 2 Results

In [14]:
# Paths
results_dir = Path('../results/indicators')
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {results_dir}")

Results directory: ../results/indicators


In [15]:
# Load descriptor consistency results from TASK 2
print("Loading descriptor consistency results from TASK 2...")

csv_path = results_dir / 'nn_descriptor_consistency.csv'

try:
    df_results = pd.read_csv(csv_path)
    print(f"  ✓ Loaded: {csv_path}")
    print(f"  Shape: {df_results.shape}")
except FileNotFoundError:
    print(f"  ❌ File not found: {csv_path}")
    print(f"  Please run TASK 2 notebook first.")
    raise

print(f"\nColumns: {df_results.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_results.head())

Loading descriptor consistency results from TASK 2...
  ✓ Loaded: ../results/indicators/nn_descriptor_consistency.csv
  Shape: (1248, 7)

Columns: ['descriptor', 'k', 'neighbors', 'nn_mean_diff', 'random_mean_diff', 'effect_ratio', 'spearman_corr']

First few rows:
          descriptor   k  neighbors  nn_mean_diff  random_mean_diff  \
0     MaxEStateIndex  10  inclusive      0.937212          2.711796   
1     MinEStateIndex  10  inclusive      0.517522          1.268156   
2  MaxAbsEStateIndex  10  inclusive      0.937212          2.711796   
3  MinAbsEStateIndex  10  inclusive      0.095543          0.232767   
4                qed  10  inclusive      0.087065          0.232067   

   effect_ratio  spearman_corr  
0      2.893471       0.280663  
1      2.450437       0.277898  
2      2.893471       0.280663  
3      2.436255       0.268873  
4      2.665433       0.272353  


## 2. Define Descriptor Families

In [16]:
def classify_descriptor(name):
    """
    Classify descriptor into family based on name pattern.
    
    RDKit descriptor naming conventions:
    - Size/Weight: MolWt, HeavyAtomCount, ExactMolWt, NumValenceElectrons
    - Surface/VSA: *VSA*, LabuteASA, TPSA
    - Topological: Chi*, Kappa*, BalabanJ, BertzCT, Ipc
    - Lipophilicity: MolLogP, MolMR
    - Charge/Electronic: *PartialCharge*, *EState*
    - Rings: *Ring*, *Aromatic*, *Aliphatic*
    - Fragments: fr_*
    - Hydrogen bonding: *HBA*, *HBD*, *Donor*, *Acceptor*
    - Rotatable: *Rotatable*
    - Saturation: *Saturat*, *Unsaturat*
    """
    name_lower = name.lower()
    
    # Fragment counts
    if name.startswith('fr_'):
        return 'Fragment Counts'
    
    # Surface area / VSA
    if 'vsa' in name_lower or name in ['LabuteASA', 'TPSA']:
        return 'Surface Area / VSA'
    
    # Topological indices
    if (name.startswith('Chi') or name.startswith('Kappa') or 
        name in ['BalabanJ', 'BertzCT', 'Ipc', 'HallKierAlpha']):
        return 'Topological'
    
    # Rings
    if 'ring' in name_lower or 'aromatic' in name_lower or 'aliphatic' in name_lower:
        return 'Rings'
    
    # Charge/Electronic
    if 'charge' in name_lower or 'estate' in name_lower:
        return 'Charge / Electronic'
    
    # Lipophilicity
    if name in ['MolLogP', 'MolMR']:
        return 'Lipophilicity'
    
    # Hydrogen bonding
    if ('hba' in name_lower or 'hbd' in name_lower or 
        'donor' in name_lower or 'acceptor' in name_lower):
        return 'H-Bonding'
    
    # Size/Weight
    if ('molwt' in name_lower or 'heavyatom' in name_lower or 
        'exactmolwt' in name_lower or 'valence' in name_lower or
        name in ['NumAtoms', 'NumHeavyAtoms', 'NumValenceElectrons']):
        return 'Size / Weight'
    
    # Rotatable bonds
    if 'rotatable' in name_lower or 'rotor' in name_lower:
        return 'Flexibility'
    
    # Saturation
    if 'saturat' in name_lower or 'unsaturat' in name_lower or 'fsp3' in name_lower:
        return 'Saturation'
    
    # Bonds
    if 'bond' in name_lower:
        return 'Bonds'
    
    # PEOE (Partial Equalization of Orbital Electronegativities)
    if name.startswith('PEOE_'):
        return 'PEOE'
    
    # SMR (Molar Refractivity)
    if name.startswith('SMR_'):
        return 'Molar Refractivity'
    
    # SlogP
    if name.startswith('SlogP_'):
        return 'SlogP'
    
    # Default
    return 'Other'

# Test classification
test_descriptors = ['MolWt', 'fr_Al_OH', 'Chi0', 'TPSA', 'NumAromaticRings', 'MolLogP']
print("Test classifications:")
for desc in test_descriptors:
    print(f"  {desc:20} → {classify_descriptor(desc)}")

Test classifications:
  MolWt                → Size / Weight
  fr_Al_OH             → Fragment Counts
  Chi0                 → Topological
  TPSA                 → Surface Area / VSA
  NumAromaticRings     → Rings
  MolLogP              → Lipophilicity


In [17]:
# Apply classification to all descriptors
print("\nClassifying descriptors into families...")

df_results['family'] = df_results['descriptor'].apply(classify_descriptor)

print(f"\nDescriptor families found:")
family_counts = df_results.groupby('family')['descriptor'].nunique().sort_values(ascending=False)
for family, count in family_counts.items():
    print(f"  {family:30} {count:3d} descriptors")

print(f"\nTotal unique descriptors: {df_results['descriptor'].nunique()}")


Classifying descriptors into families...

Descriptor families found:
  Fragment Counts                 85 descriptors
  Surface Area / VSA              59 descriptors
  Topological                     19 descriptors
  Other                           17 descriptors
  Charge / Electronic              8 descriptors
  Rings                            8 descriptors
  Size / Weight                    5 descriptors
  H-Bonding                        2 descriptors
  Lipophilicity                    2 descriptors
  Saturation                       2 descriptors
  Flexibility                      1 descriptors

Total unique descriptors: 208


## 3. Aggregate by Family

In [18]:
# Filter for k=50, exclusive (our primary analysis)
print("\nAggregating by family (k=50, exclusive)...")

df_k50_excl = df_results[
    (df_results['k'] == 50) & 
    (df_results['neighbors'] == 'exclusive')
].copy()

# Compute family-level statistics
family_stats = df_k50_excl.groupby('family').agg({
    'descriptor': 'count',
    'effect_ratio': ['median', 'mean', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)],
    'spearman_corr': ['median', 'mean'],
    'nn_mean_diff': ['median', 'mean'],
    'random_mean_diff': ['median', 'mean']
}).round(4)

# Flatten column names
family_stats.columns = ['_'.join(col).strip() if col[1] else col[0] for col in family_stats.columns.values]

# Rename columns
family_stats = family_stats.rename(columns={
    'descriptor_count': 'n_descriptors',
    'effect_ratio_median': 'effect_ratio_median',
    'effect_ratio_mean': 'effect_ratio_mean',
    'effect_ratio_<lambda_0>': 'effect_ratio_q25',
    'effect_ratio_<lambda_1>': 'effect_ratio_q75',
    'spearman_corr_median': 'spearman_median',
    'spearman_corr_mean': 'spearman_mean',
    'nn_mean_diff_median': 'nn_diff_median',
    'nn_mean_diff_mean': 'nn_diff_mean',
    'random_mean_diff_median': 'random_diff_median',
    'random_mean_diff_mean': 'random_diff_mean'
})

# Compute IQR
family_stats['effect_ratio_iqr'] = family_stats['effect_ratio_q75'] - family_stats['effect_ratio_q25']

# Sort by median effect ratio
family_stats = family_stats.sort_values('effect_ratio_median', ascending=False)

print("\n✓ Family statistics computed")
print(f"\nColumns: {family_stats.columns.tolist()}")


Aggregating by family (k=50, exclusive)...

✓ Family statistics computed

Columns: ['n_descriptors', 'effect_ratio_median', 'effect_ratio_mean', 'effect_ratio_q25', 'effect_ratio_q75', 'spearman_median', 'spearman_mean', 'nn_diff_median', 'nn_diff_mean', 'random_diff_median', 'random_diff_mean', 'effect_ratio_iqr']


## 4. Save Results

In [19]:
# Save to CSV
output_path = results_dir / 'descriptor_family_summary.csv'
family_stats.to_csv(output_path)

print(f"✅ Saved family summary to: {output_path}")

✅ Saved family summary to: ../results/indicators/descriptor_family_summary.csv


## 5. Display Summary Table

In [20]:
# Print formatted summary
print("\n" + "="*90)
print("DESCRIPTOR FAMILY SUMMARY (k=50, exclusive)")
print("="*90)

print(f"\n{'Family':<30} {'N':>4} {'Median':>8} {'IQR':>8} {'Mean':>8} {'Spearman':>10}")
print(f"{'':30} {'':4} {'Ratio':>8} {'':>8} {'Ratio':>8} {'Corr':>10}")
print("-" * 90)

for family, row in family_stats.iterrows():
    print(f"{family:<30} {row['n_descriptors']:4.0f} "
          f"{row['effect_ratio_median']:8.3f} {row['effect_ratio_iqr']:8.3f} "
          f"{row['effect_ratio_mean']:8.3f} {row['spearman_median']:10.4f}")

print("\n" + "="*90)
print("Interpretation:")
print("  - Median Ratio: Typical effect ratio for descriptors in this family")
print("  - IQR: Interquartile range (spread of ratios within family)")
print("  - High ratio (>1): Family is well-captured by embeddings")
print("  - Low ratio (<1): Family is poorly captured by embeddings")
print("  - Spearman Corr: Positive = good alignment (similar embeddings → similar descriptors)")
print("="*90)


DESCRIPTOR FAMILY SUMMARY (k=50, exclusive)

Family                            N   Median      IQR     Mean   Spearman
                                       Ratio             Ratio       Corr
------------------------------------------------------------------------------------------
Size / Weight                     5    2.113    0.009    2.114     0.0982
Topological                      19    2.041    0.269    1.925     0.0802
Lipophilicity                     2    1.728    0.315    1.728     0.0756
Saturation                        2    1.660    0.013    1.660     0.0359
Rings                             8    1.649    0.133    1.676     0.0586
H-Bonding                         2    1.571    0.232    1.571     0.0582
Other                            17    1.533    0.320    1.455     0.0776
Flexibility                       1    1.484    0.000    1.484     0.0650
Surface Area / VSA               59    1.411    0.184    1.398     0.0565
Charge / Electronic               8    1.404    0

## 6. Top Descriptors per Family

In [21]:
# Show top 3 descriptors per family
print("\n" + "="*90)
print("TOP 3 DESCRIPTORS PER FAMILY (by effect ratio)")
print("="*90)

for family in family_stats.index[:5]:  # Top 5 families
    print(f"\n{family}:")
    print(f"  {'Descriptor':<35} {'Effect Ratio':>15} {'Spearman':>12}")
    print("  " + "-" * 62)
    
    family_descriptors = df_k50_excl[
        df_k50_excl['family'] == family
    ].sort_values('effect_ratio', ascending=False).head(3)
    
    for _, row in family_descriptors.iterrows():
        print(f"  {row['descriptor']:<35} {row['effect_ratio']:15.4f} {row['spearman_corr']:12.4f}")

print("\n" + "="*90)


TOP 3 DESCRIPTORS PER FAMILY (by effect ratio)

Size / Weight:
  Descriptor                             Effect Ratio     Spearman
  --------------------------------------------------------------
  NumValenceElectrons                          2.1277       0.0941
  HeavyAtomCount                               2.1216       0.0942
  ExactMolWt                                   2.1127       0.0987

Topological:
  Descriptor                             Effect Ratio     Spearman
  --------------------------------------------------------------
  Chi2n                                        2.1297       0.0867
  Chi1n                                        2.1262       0.0912
  Chi3n                                        2.1204       0.0802

Lipophilicity:
  Descriptor                             Effect Ratio     Spearman
  --------------------------------------------------------------
  MolMR                                        2.0434       0.0894
  MolLogP                                

## 7. Cross-k Analysis

In [22]:
# Compare family performance across k values
print("\nFamily performance across k values (exclusive)...\n")

df_excl = df_results[df_results['neighbors'] == 'exclusive'].copy()

# Pivot: family × k
family_by_k = df_excl.pivot_table(
    index='family',
    columns='k',
    values='effect_ratio',
    aggfunc='median'
).round(3)

# Add mean across k
family_by_k['mean_k'] = family_by_k.mean(axis=1)
family_by_k = family_by_k.sort_values('mean_k', ascending=False)

print("Effect Ratio (median) by Family and k:")
print(family_by_k)


Family performance across k values (exclusive)...

Effect Ratio (median) by Family and k:
k                       10     50    100    mean_k
family                                            
Size / Weight        2.841  2.113  1.947  2.300333
Topological          2.741  2.041  1.893  2.225000
Lipophilicity        2.285  1.728  1.612  1.875000
Saturation           2.206  1.660  1.559  1.808333
Rings                2.212  1.649  1.542  1.801000
H-Bonding            2.043  1.571  1.482  1.698667
Other                1.991  1.533  1.435  1.653000
Flexibility          1.959  1.484  1.390  1.611000
Charge / Electronic  1.861  1.404  1.319  1.528000
Surface Area / VSA   1.820  1.411  1.335  1.522000
Fragment Counts      1.522  1.210  1.156  1.296000


## 8. Task Complete

In [23]:
print("\n" + "="*80)
print("TASK 3 — Descriptor Family Aggregation COMPLETE")
print("="*80)
print(f"\nResults saved:")
print(f"  - CSV: results/indicators/descriptor_family_summary.csv")
print(f"\nKey findings:")
print(f"  - Total families: {len(family_stats)}")
print(f"  - Best family: {family_stats.index[0]} (median ratio: {family_stats.iloc[0]['effect_ratio_median']:.3f})")
print(f"  - Worst family: {family_stats.index[-1]} (median ratio: {family_stats.iloc[-1]['effect_ratio_median']:.3f})")
print(f"\nNext steps:")
print(f"  - Use these results to interpret which molecular properties are well-captured")
print(f"  - Generate visualizations (heatmaps, bar charts)")
print(f"  - Compare with TASK 4 (if applicable)")
print("="*80)


TASK 3 — Descriptor Family Aggregation COMPLETE

Results saved:
  - CSV: results/indicators/descriptor_family_summary.csv

Key findings:
  - Total families: 11
  - Best family: Size / Weight (median ratio: 2.113)
  - Worst family: Fragment Counts (median ratio: 1.210)

Next steps:
  - Use these results to interpret which molecular properties are well-captured
  - Generate visualizations (heatmaps, bar charts)
  - Compare with TASK 4 (if applicable)
